# 🐾 Dog Skin Disease Classifier
**Upload an image → Get disease prediction with confidence**

Classes: `Dermatitis | Fungal Infections | Healthy | Hypersensitivity | Demodicosis | Ringworm`

In [ ]:
# ── CELL 1: Install dependencies ─────────────────────────────────────────────
!pip install tensorflow scikit-learn opencv-python-headless matplotlib joblib -q

In [ ]:
# ── CELL 2: Mount Google Drive (your models must be uploaded here) ────────────
from google.colab import drive
drive.mount('/content/drive')

# Set this to where you uploaded your models folder in Drive
# e.g. if you put it at: My Drive/AnimalDisease/models/
MODEL_DIR = '/content/drive/MyDrive/AnimalDisease/models'

import os
print('Files in model dir:', os.listdir(MODEL_DIR))

In [ ]:
# ── CELL 3: Imports & Load Models ────────────────────────────────────────────
import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import numpy as np
import cv2
import joblib
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre

IMG_SIZE     = (224, 224)
W_SVM, W_DNN = 0.4, 0.6

# Exact alphabetical order Keras uses when loading dataset
CLASS_NAMES = [
    'Dermatitis',
    'Fungal Infections',
    'Healthy',
    'Hypersensitivity',
    'Demodicosis',
    'Ringworm'
]

CLASS_COLORS = {
    'Dermatitis':        (231, 76,  60),
    'Fungal Infections': (142, 68, 173),
    'Healthy':           (39, 174,  96),
    'Hypersensitivity':  (230, 126, 34),
    'Demodicosis':       (41, 128, 185),
    'Ringworm':          (192, 57,  43)
}

print('Loading models...')
resnet = ResNet50(weights='imagenet', include_top=False,
                  pooling='avg', input_shape=(224, 224, 3))
mobile = tf.keras.models.load_model(os.path.join(MODEL_DIR, 'mobilenet_best.h5'))
svm    = joblib.load(os.path.join(MODEL_DIR, 'svm_model.pkl'))
scaler = joblib.load(os.path.join(MODEL_DIR, 'scaler.pkl'))
print('✅ All models loaded!')

In [ ]:
# ── CELL 4: Prediction Pipeline ───────────────────────────────────────────────
def predict_disease(img_path):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        raise ValueError(f'Cannot read image: {img_path}')

    # ResNet50 → SVM
    r = np.expand_dims(resnet_pre(cv2.resize(img_bgr, IMG_SIZE).astype(np.float32)), 0)
    svm_proba = svm.predict_proba(scaler.transform(resnet.predict(r, verbose=0)))[0]

    # MobileNetV2
    m = np.expand_dims(mob_pre(cv2.resize(img_bgr, IMG_SIZE).astype(np.float32)), 0)
    dnn_proba = mobile.predict(m, verbose=0)[0]

    # Weighted fusion
    final   = W_SVM * svm_proba + W_DNN * dnn_proba
    idx     = int(np.argmax(final))
    conf    = float(np.max(final))
    return CLASS_NAMES[idx], conf, img_bgr


def show_result(img_path):
    disease, confidence, img_bgr = predict_disease(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    out     = img_bgr.copy()
    h, w    = out.shape[:2]

    # Draw semi-transparent banner at bottom
    r, g, b   = CLASS_COLORS.get(disease, (44, 62, 80))
    bgr_color = (b, g, r)
    banner_h  = int(h * 0.20)
    overlay   = out.copy()
    cv2.rectangle(overlay, (0, h - banner_h), (w, h), bgr_color, -1)
    cv2.addWeighted(overlay, 0.82, out, 0.18, 0, out)

    # Text
    font  = cv2.FONT_HERSHEY_DUPLEX
    scale = max(0.7, w / 600)
    thick = max(1, int(w / 350))
    cv2.putText(out, f'Disease: {disease}',
                (int(w*0.03), h - int(banner_h*0.58)),
                font, scale * 1.1, (255,255,255), thick+1, cv2.LINE_AA)
    cv2.putText(out, f'Confidence: {confidence*100:.1f}%',
                (int(w*0.03), h - int(banner_h*0.15)),
                font, scale, (220,220,220), thick, cv2.LINE_AA)

    # Display
    plt.figure(figsize=(7, 7))
    plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    print(f'\n  Predicted  : {disease}')
    print(f'  Confidence : {confidence*100:.2f}%')
    if confidence < 0.55:
        print('  ⚠️  Low confidence — consult a vet.')

print('✅ Pipeline ready. Run next cell to upload and predict.')

In [ ]:
# ── CELL 5: Upload Image & Predict ────────────────────────────────────────────
from google.colab import files

print('Select an image to upload:')
uploaded = files.upload()
img_path = list(uploaded.keys())[0]

show_result(img_path)